# 12. Creating Issues, Worklogs and Attachments

Scripts don't just read Jira, they create work in it: a monitoring check opens an incident, a form submission becomes a ticket, a CSV of requests becomes a backlog.

> **Before you run this notebook:** create an API token (topic 08) and set `JIRA_SERVER`, `JIRA_EMAIL` and `JIRA_API_TOKEN` as environment variables, or in Colab's **Secrets** panel. The outputs shown were recorded on a practice Jira Cloud site with a project called **myproject** (key `MYP`). Your issue keys and data will differ.

In [1]:
import os
from jira import JIRA

jira = JIRA(
    server=os.environ["JIRA_SERVER"],                 # https://your-domain.atlassian.net
    basic_auth=(os.environ["JIRA_EMAIL"], os.environ["JIRA_API_TOKEN"]),
)

## Create One Issue

In [2]:
new_issue = jira.create_issue(
    project="MYP",
    summary="This is a test issue from Python",
    description="Detailed description of the issue",
    issuetype={"name": "Bug"},
)

print(f"Issue {new_issue.key} created successfully!")

Issue MYP-36 created successfully!


| Argument | Value |
|---|---|
| `project` | The project **key** |
| `summary` | The title (required) |
| `description` | The body text |
| `issuetype` | A dict with the type **name**: `Bug`, `Task`, `Story` |
| `priority` | A dict with the priority name: `Highest`, `High`, `Medium`, `Low`, `Lowest` |

The same call also accepts everything as one `fields` dictionary, which is handy when the fields are built up in code:

```python
issue_dict = {
    "project": {"key": "MYP"},
    "summary": "This is a test issue from Python",
    "description": "Detailed description of the issue",
    "issuetype": {"name": "Bug"},
}
new_issue = jira.create_issue(fields=issue_dict)
```

## Create 20 Issues with Random Settings

Useful for filling a practice project with data to search and report on:

In [3]:
import random

issue_types = ["Bug", "Task", "Story"]
priorities = ["Low", "Lowest", "High", "Highest"]

for i in range(1, 21):
    issue_type = random.choice(issue_types)
    priority = random.choice(priorities)

    new_issue = jira.create_issue(
        project="MYP",
        summary=f"Automated Issue #{i}",
        description=f"This is an automated test issue number {i} with various configurations.",
        issuetype={"name": issue_type},
        priority={"name": priority},
    )
    print(f"Issue {new_issue.key} created successfully with priority {priority} and type {issue_type}!")

Issue MYP-37 created successfully with priority Lowest and type Bug!
Issue MYP-38 created successfully with priority Highest and type Bug!
Issue MYP-39 created successfully with priority Low and type Bug!
Issue MYP-40 created successfully with priority Highest and type Task!
Issue MYP-41 created successfully with priority Lowest and type Bug!
Issue MYP-42 created successfully with priority Highest and type Task!
Issue MYP-43 created successfully with priority Low and type Story!
Issue MYP-44 created successfully with priority Lowest and type Task!
Issue MYP-45 created successfully with priority High and type Task!
Issue MYP-46 created successfully with priority Highest and type Bug!
Issue MYP-47 created successfully with priority Low and type Task!
Issue MYP-48 created successfully with priority Low and type Task!
Issue MYP-49 created successfully with priority Lowest and type Task!
Issue MYP-50 created successfully with priority Low and type Bug!
Issue MYP-51 created successfully with

> Run bulk scripts on a **practice project or development site** only. There is no undo button for 20 new tickets.

## Bulk Create from a CSV File

[`data/sample_jira_issues.csv`](data/sample_jira_issues.csv) holds a small backlog:

```text
Summary,Project Key,Issue Type,Description,Priority
Sample Task 1,MYP,Task,This is a sample task,Medium
Sample Bug 1,MYP,Bug,This is a sample bug,High
...
Research Task,KAN,Task,Investigate a new technology,High
```

Jira Cloud's default priorities are **Highest, High, Medium, Low, Lowest**. The file uses `Critical` for two rows, which doesn't exist by default, so the script maps it to `Highest`.

In [4]:
import csv

PRIORITY_MAP = {"Critical": "Highest"}

with open("data/sample_jira_issues.csv", newline="") as f:
    for row in csv.DictReader(f):
        priority = PRIORITY_MAP.get(row["Priority"], row["Priority"])
        new_issue = jira.create_issue(
            project=row["Project Key"],
            summary=row["Summary"],
            description=row["Description"],
            issuetype={"name": row["Issue Type"]},
            priority={"name": priority},
        )
        print(f"{new_issue.key}: {row['Summary']} ({row['Issue Type']}, {priority})")

> Jira also has a built-in importer: **Settings (gear) > System > External system import > CSV**. Use it for one-off migrations, and Python when the import must run regularly or needs logic.

## Time Tracking Estimates

In [5]:
new_issue = jira.create_issue(
    project="MYP",
    summary="Issue with Time Tracking",
    description="This issue includes time tracking details.",
    issuetype={"name": "Task"},
    priority={"name": "High"},
    timetracking={"originalEstimate": "5h", "remainingEstimate": "3h"},
)

print(f"Issue {new_issue.key} created successfully with time tracking!")

Issue MYP-57 created successfully with time tracking!


Time values use Jira's units: `30m`, `5h`, `2d`, `1w`. Time tracking must be enabled for the project's screens.

## Finding a User's Account ID

Jira Cloud identifies people by **account ID**, not by name or email, so you need the ID to assign an issue.

In [6]:
search_name = "Ahmad Hassan"
users = jira.search_users(query=search_name)

for user in users:
    print(f"Name: {user.displayName}, Account ID: {user.accountId}")

Name: Ahmad Hassan, Account ID: <ahmad-account-id>
Name: Hassan, Account ID: <hassan-account-id>


The search matched two people, because the query matches parts of names. Pick the right one, then store the ID in a variable or a config file.

## Create an Issue with an Assignee

In [7]:
hassan_account_id = "<hassan-account-id>"     # from search_users()

new_issue = jira.create_issue(
    project="MYP",
    summary="Task Assigned to Hassan",
    description="This task is assigned to Hassan using the account ID.",
    issuetype={"name": "Task"},
    priority={"name": "High"},
    assignee={"accountId": hassan_account_id},
)

print(f"Issue {new_issue.key} created and assigned to Hassan!")

Issue MYP-58 created and assigned to Hassan!


- The **reporter** is set automatically to the account that owns the API token.
- To change the assignee later: `new_issue.update(assignee={"accountId": hassan_account_id})` or `jira.assign_issue(new_issue, hassan_account_id)`.

## Log Work

In [8]:
issue_key = "MYP-73"
jira.add_worklog(issue=issue_key, timeSpent="2h", comment="Worked on the initial setup.")

print(f"Worklog added successfully to {issue_key}!")

Worklog added successfully to MYP-73!


## Add an Attachment

In [9]:
with open("data/sample.txt", "rb") as f:
    attachment = jira.add_attachment(issue="MYP-73", attachment=f)
print(attachment)

<JIRA Attachment: filename='sample.txt', id='10004', mimeType='text/plain'>


Open the file in **binary mode** (`"rb"`). Attach screenshots, log files or reports the same way.

## Moving an Issue Through Its Workflow

Status isn't a normal field: you change it by running a **transition** (topic 07).

In [10]:
issue = jira.issue("MYP-73")

for t in jira.transitions(issue):
    print(t["id"], t["name"])

In [11]:
jira.transition_issue(issue, "In Progress")      # the transition name, or its id
print(jira.issue("MYP-73").fields.status.name)

Transition names depend on the workflow, so list them first. A transition that isn't allowed from the current status raises an error.

## Practice

Create a **Story** called *"Dashboard shows model accuracy"* with priority **Medium**, a 1-day estimate, assigned to yourself, then add the comment *"Created by the practice script"*.

**Solution:**

In [12]:
me = jira.myself()["accountId"]

story = jira.create_issue(
    project="MYP",
    summary="Dashboard shows model accuracy",
    description="Show the latest validation accuracy of each model on the team dashboard.",
    issuetype={"name": "Story"},
    priority={"name": "Medium"},
    timetracking={"originalEstimate": "1d"},
    assignee={"accountId": me},
)
jira.add_comment(story, "Created by the practice script")
print(story.key, story.fields.summary)